## Evaluate Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework.

Steps:
1. Load dataset
2. Dataset statistics
3. Define the Presidio Analyzer
4. Run predictions
5. Review and adjust entity mapping
6. Evaluate
7. Results and error analysis

For an example with a custom Presidio instance, see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb).

In [1]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator
!pip install -e ..

Obtaining file:///home/noaevag/presidio-research
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for presidio_evaluator (pyproject.toml) ... done
  Created wheel for presidio_evaluator: filename=presidio_evaluator-0.3-py3-none-any.whl size=12409 sha256=1cfb51067e8de9f9c11f420120fd4bc94adf5f2b2fa0fc4f12056a0daa4741b7
  Stored in directory: /tmp/pip-ephem-wheel-cache-fy0c62h2/wheels/7e/f0/a4/6adf642173042e249a26e8496268f8cc5a0a6275c632844502
Successfully built presidio_evaluator
  Attempting uninstall: presidio_evaluator
    Found existing installation: presidio_evaluator 0.3
    Uninstalling presidio_evaluator-0.3:
      Successfully uninstalled presidio_evaluator-0.3


In [1]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd
from presidio_analyzer import AnalyzerEngine

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import CanonicalMapper
from presidio_evaluator.evaluation import ModelError, Plotter, SpanEvaluator
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import PresidioAnalyzerWrapper

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2

## 1. Load dataset from file

In [2]:
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))
print(len(dataset))

tokenizing input:   0%|          | 0/1500 [00:00<?, ?it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:10<00:00, 137.09it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [3]:
def get_entity_counts(dataset: list[InputSample]) -> Counter:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

## 2. Simple dataset statistics

In [4]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print(
    "\nMin and max number of tokens in dataset: "
    f"Min: {min([len(sample.tokens) for sample in dataset])}, "
    f"Max: {max([len(sample.tokens) for sample in dataset])}"
)

print(
    f"Min and max sentence length in dataset: "
    f"Min: {min([len(sample.full_text) for sample in dataset])}, "
    f"Max: {max([len(sample.full_text) for sample in dataset])}"
)

print("\nExample InputSample:")
print(dataset[0])

Count per entity:
[('O', 19626), ('STREET_ADDRESS', 3071), ('PERSON', 1369), ('GPE', 521),
 ('ORGANIZATION', 504), ('PHONE_NUMBER', 350), ('DATE_TIME', 219),
 ('TITLE', 142), ('CREDIT_CARD', 136), ('US_SSN', 80), ('AGE', 74), ('NRP', 55),
 ('ZIP_CODE', 50), ('EMAIL_ADDRESS', 49), ('DOMAIN_NAME', 37),
 ('IP_ADDRESS', 22), ('IBAN_CODE', 21), ('US_DRIVER_LICENSE', 9)]

Min and max number of tokens in dataset: Min: 3, Max: 78
Min and max sentence length in dataset: Min: 9, Max: 407

Example InputSample:
Full text: The address of Persint is 6750 Koskikatu 25 Apt. 864
Artilleros
, CO
 Uruguay 64677
Spans: [Span(type: STREET_ADDRESS, value: 6750 Koskikatu 25 Apt. 864
Artilleros
, CO
 Uruguay 64677, char_span: [26: 83]), Span(type: ORGANIZATION, value: Persint, char_span: [15: 22])]



In [ ]:
print("A few examples sentences containing each entity:\n")
for entity in entity_counts.keys():
    samples = [sample for sample in dataset if entity in set(sample.tags)]
    if len(samples) > 1 and entity != "O":
        print(
            f"Entity: <{entity}> two example sentences:\n"
            f"\n1) {samples[0].full_text}"
            f"\n2) {samples[1].full_text}"
            f"\n------------------------------------\n"
        )

## 3. Define the Presidio Analyzer

Using Presidio with default parameters (not recommended for production). For a customised example see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb).

**Note:** The dataset may use different entity labels than Presidio (e.g. `STREET_ADDRESS` vs `LOCATION`). We'll align them in section 5 after running predictions.

In [5]:
# Loading the vanilla Analyzer Engine, with the default NER model.
analyzer_engine = AnalyzerEngine(default_score_threshold=0.4)

pprint("Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint(
    [
        rec.name
        for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)
    ],
    compact=True,
)

print("\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['MEDICAL_LICENSE', 'US_PASSPORT', 'PHONE_NUMBER', 'MAC_ADDRESS', 'CREDIT_CARD',
 'US_SSN', 'US_BANK_NUMBER', 'US_DRIVER_LICENSE', 'NRP', 'US_ITIN', 'URL',
 'PERSON', 'EMAIL_ADDRESS', 'IBAN_CODE', 'IP_ADDRESS', 'UK_NHS', 'CRYPTO',
 'DATE_TIME', 'LOCATION']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsLicenseRecognizer',
 'UsItinRecognizer', 'UsPassportRecognizer', 'UsSsnRecognizer', 'NhsRecognizer',
 'CryptoRecognizer', 'DateRecognizer', 'EmailRecognizer', 'IbanRecognizer',
 'IpRecognizer', 'MedicalLicenseRecognizer', 'MacAddressRecognizer',
 'PhoneRecognizer', 'UrlRecognizer', 'SpacyRecognizer']

Loaded NER models:
[{'lang_code': 'en', 'model_name': 'en_core_web_lg'}]


In [6]:
# Wrap the analyzer
wrapped_analyzer = PresidioAnalyzerWrapper(analyzer_engine=analyzer_engine)

# Set up the experiment tracker and log model + dataset params
experiment = get_experiment_tracker()
params = {"dataset_name": dataset_name, "model_name": wrapped_analyzer.name}
params.update(wrapped_analyzer.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)

--------
Entities supported by this Presidio Analyzer instance:
MEDICAL_LICENSE, US_PASSPORT, PHONE_NUMBER, MAC_ADDRESS, CREDIT_CARD, US_SSN, US_BANK_NUMBER, US_DRIVER_LICENSE, NRP, US_ITIN, URL, PERSON, EMAIL_ADDRESS, IBAN_CODE, IP_ADDRESS, UK_NHS, CRYPTO, DATE_TIME, LOCATION


## 4. Run predictions

Run the model on the full dataset. Returns a 5-column DataFrame (`sentence_id`, `token`, `annotation`, `prediction`, `start_indices`).

In [7]:
%%time
results_df = wrapped_analyzer.predict_dataset(dataset)
results_df.head()

CPU times: user 14.3 s, sys: 135 ms, total: 14.4 s
Wall time: 14.1 s


,sentence_id,token,annotation,prediction,start_indices
0,0,The,O,O,0
1,0,address,O,O,4
2,0,of,O,O,12
3,0,Persint,ORGANIZATION,O,15
4,0,is,O,O,23


## 5. Review entity mapping

`CanonicalMapper` auto-resolves entity labels by comparing the `annotation` and `prediction` columns in the results DataFrame. Run the cell below to see the auto-resolved mapping.

If any labels look wrong, update the overrides in the adjustment cell and re-run.

In [8]:
# Create the mapper — labels are extracted and auto-resolved later by get_mapped_results_dataframe()
mapper = CanonicalMapper()

In [9]:
# Auto-resolve labels from annotation + prediction columns and inspect the result
mapped_df = mapper.get_mapped_results_dataframe(results_df)
mapper.render_html()
mapped_df.head()

/tmp/ipykernel_380926/2848899579.py:2: UserWarning: Some annotations and predictions resolve to different canonical entities: 'AGE' vs 'DATE_TIME', 'CREDIT_CARD' vs 'DATE_TIME', 'CREDIT_CARD' vs 'PHONE_NUMBER', 'DOMAIN_NAME' vs 'URL', 'GPE' vs 'DATE_TIME', 'GPE' vs 'LOCATION', 'GPE' vs 'NRP', 'GPE' vs 'PERSON', 'IP_ADDRESS' vs 'PERSON', 'NRP' vs 'LOCATION', 'NRP' vs 'PERSON', 'ORGANIZATION' vs 'LOCATION', 'ORGANIZATION' vs 'NRP', 'ORGANIZATION' vs 'PERSON', 'PERSON' vs 'LOCATION', 'PERSON' vs 'NRP', 'PHONE_NUMBER' vs 'DATE_TIME', 'STREET_ADDRESS' vs 'DATE_TIME', 'STREET_ADDRESS' vs 'LOCATION', 'STREET_ADDRESS' vs 'NRP', 'STREET_ADDRESS' vs 'PERSON', 'STREET_ADDRESS' vs 'PHONE_NUMBER', 'TITLE' vs 'PERSON', 'US_DRIVER_LICENSE' vs 'PHONE_NUMBER', 'ZIP_CODE' vs 'DATE_TIME', 'ZIP_CODE' vs 'PERSON'. This will count as false negatives/positives. To fix this: (a) call get_mapped_results_dataframe(results_df, hierarchy=2) to use a broader mapping level where both resolve to the same entity, or 

Raw Label,Tier,Canonical,Score
ORGANIZATION,EXACT,ORGANIZATION,—
STREET_ADDRESS,EXACT,ADDRESS,—
PERSON,EXACT,PERSON,—
DATE_TIME,EXACT,DATE_TIME,—
GPE,EXACT,GPE,—
CREDIT_CARD,EXACT,FINANCIAL,—
US_SSN,EXACT,SSN,—
US_DRIVER_LICENSE,EXACT,DRIVER_LICENSE,—
AGE,EXACT,AGE,—
ZIP_CODE,EXACT,ADDRESS,—


,sentence_id,token,annotation,prediction,start_indices
0,0,The,O,O,0
1,0,address,O,O,4
2,0,of,O,O,12
3,0,Persint,ORGANIZATION,O,15
4,0,is,O,O,23


### Adjust mapping if needed

If labels are resolved incorrectly or should be suppressed, update `mapper.map({...})` in the cell below and re-run, then re-run the mapping cell above to see the updated result.

> For more detail on how entity mapping works, see [notebook 6 — Interactive Entity Mapping](6_Interactive_Entity_Mapping.ipynb).

In [10]:
# Edit overrides below if anything needs correcting, then re-run this cell and the mapping cell above
mapper.map({
    "AGE": "DATE_TIME",
    "TITLE": "PERSON",
    "ORGANIZATION": None,   # suppress — model doesn't detect it
})
# Re-apply mapping with updated overrides
mapped_df = mapper.get_mapped_results_dataframe(results_df)
mapper.render_html()
mapped_df.head()

/tmp/ipykernel_380926/4259044786.py:8: UserWarning: Some annotations and predictions resolve to different canonical entities: 'CREDIT_CARD' vs 'DATE_TIME', 'CREDIT_CARD' vs 'PHONE_NUMBER', 'DOMAIN_NAME' vs 'URL', 'GPE' vs 'DATE_TIME', 'GPE' vs 'LOCATION', 'GPE' vs 'NRP', 'GPE' vs 'PERSON', 'IP_ADDRESS' vs 'PERSON', 'NRP' vs 'LOCATION', 'NRP' vs 'PERSON', 'ORGANIZATION' vs 'LOCATION', 'ORGANIZATION' vs 'NRP', 'ORGANIZATION' vs 'PERSON', 'PERSON' vs 'LOCATION', 'PERSON' vs 'NRP', 'PHONE_NUMBER' vs 'DATE_TIME', 'STREET_ADDRESS' vs 'DATE_TIME', 'STREET_ADDRESS' vs 'LOCATION', 'STREET_ADDRESS' vs 'NRP', 'STREET_ADDRESS' vs 'PERSON', 'STREET_ADDRESS' vs 'PHONE_NUMBER', 'US_DRIVER_LICENSE' vs 'PHONE_NUMBER', 'ZIP_CODE' vs 'DATE_TIME', 'ZIP_CODE' vs 'PERSON'. This will count as false negatives/positives. To fix this: (a) call get_mapped_results_dataframe(results_df, hierarchy=2) to use a broader mapping level where both resolve to the same entity, or (b) call mapper.map({'<label>': '<canonical

Raw Label,Tier,Canonical,Score
STREET_ADDRESS,EXACT,ADDRESS,—
PERSON,EXACT,PERSON,—
DATE_TIME,EXACT,DATE_TIME,—
GPE,EXACT,GPE,—
CREDIT_CARD,EXACT,FINANCIAL,—
US_SSN,EXACT,SSN,—
US_DRIVER_LICENSE,EXACT,DRIVER_LICENSE,—
AGE,MANUAL,DATE_TIME,—
ZIP_CODE,EXACT,ADDRESS,—
TITLE,MANUAL,PERSON,—


,sentence_id,token,annotation,prediction,start_indices
0,0,The,O,O,0
1,0,address,O,O,4
2,0,of,O,O,12
3,0,Persint,ORGANIZATION,O,15
4,0,is,O,O,23


In [11]:
experiment.log_parameter("entity_mappings", json.dumps(mapper.get_mapping()))

## 6. Evaluate

In [12]:
evaluator = SpanEvaluator(iou_threshold=0.75)


In [13]:
# Per-entity type evaluation
results = evaluator.calculate_score_on_df(results_df=mapped_df)

# Global PII precision/recall/F (all entity types collapsed to PII vs O)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()


saving experiment data to /home/noaevag/presidio-research/notebooks/experiment_20260329-232605.json


## 7. Evaluate results

In [14]:
# Plot output
plotter = Plotter(
    results=results, model_name=wrapped_analyzer.name, display_mode="interactive", beta=2
)

plotter.plot_scores()

In [15]:
pprint(
    {
        "PII F": results.pii_f,
        "PII recall": results.pii_recall,
        "PII precision": results.pii_precision,
    }
)

{'PII F': 0.661352229111308,
 'PII precision': 0.7332242225859247,
 'PII recall': 0.6455331412103746}


## 8. Error analysis

Now let's look into results to understand what's behind the metrics we're getting.
Note that evaluation is never perfect. Some things to consider:
1. There's often a mismatch between the annotated span and the predicted span, which isn't necessarily a mistake. For example: `<Southern France>` compared with `Southern <France>`. In the second text, the word `Southern` was not annotated/predicted as part of the entity, but that's not necessarily an error.
1. The synthetic dataset used here isn't representative of a real dataset. Consider using more realistic datasets for evaluation

In [16]:
plotter.plot_confusion_matrix(entities=entities, confmatrix=confmatrix)

In [17]:
plotter.plot_most_common_tokens()

### 8a. False positives
#### Most common false positive tokens:

In [18]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('norway', 18),
 ('estonia', 17),
 ('8', 15),
 ('sunday labor', 13),
 ('cyprus', 11),
 ('hungary', 10),
 ('greek', 10),
 ('france', 10),
 ('sweden', 10),
 ('czech republic', 9)]
---------------
Example sentence with each FP token:
	- Norway (`norway` pred as LOCATION)
	- Estonia (`estonia` pred as LOCATION)
	- 8 + years (`8` pred as DATE_TIME)
	- the last Sunday before Labor Day (`sunday labor` pred as DATE_TIME)
	- Cyprus (`cyprus` pred as LOCATION)
	- Hungary (`hungary` pred as LOCATION)
	- Greek (`greek` pred as NATIONALITY)
	- France (`france` pred as LOCATION)
	- Sweden (`sweden` pred as LOCATION)
	- Czech Republic (`czech republic` pred as LOCATION)


[('norway', 18),
 ('estonia', 17),
 ('8', 15),
 ('sunday labor', 13),
 ('cyprus', 11),
 ('hungary', 10),
 ('greek', 10),
 ('france', 10),
 ('sweden', 10),
 ('czech republic', 9)]

#### More FP analysis

In [19]:
# Get false positives for PERSON entity
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity="PERSON")
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,J Gelencsér ג€,j gelencsér ג€,O,PERSON
1,"Answer:""Tube Snake Boogie","answer:""tube snake boogie",O,PERSON
2,Faina D. Yefremova 's,faina d. yefremova,O,PERSON
3,Sara Schwarz \n,sara schwarz \n,O,PERSON
4,Rua,rua,O,PERSON
5,Verdafero Eklund Michael Hodge,verdafero eklund michael hodge,O,PERSON
6,Verdafero Eklund Michael Hodge,verdafero eklund michael hodge,O,PERSON
7,Király u.,király u.,O,PERSON
8,Kaczmarek Bonifacy Kaczmarek,kaczmarek bonifacy kaczmarek,O,PERSON
9,Kaczmarek Bonifacy Kaczmarek,kaczmarek bonifacy kaczmarek,O,PERSON


### 8b. False negatives (FN)

#### Most common false negative examples (should be predicted as entity but wasn't):

In [20]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('estonia', 17),
 ('norway', 16),
 ('greenlander', 11),
 ('canada', 10),
 ('czech republic', 9),
 ('france', 8),
 ('united states', 8),
 ('iceland', 8),
 ('sweden', 8),
 ('hungary', 7),
 ('tunisia', 6),
 ('finland', 6),
 ('uruguay', 6),
 ('austria', 5),
 ('poland', 5)]
---------------
Example sentence with each FN token:
	- Estonia (`estonia` annotated as GPE)
	- Norway (`norway` annotated as GPE)
	- Greenlander (`greenlander` annotated as GPE)
	- Canada (`canada` annotated as GPE)
	- Czech Republic (`czech republic` annotated as GPE)
	- France (`france` annotated as GPE)
	- United States (`united states` annotated as GPE)
	- Iceland (`iceland` annotated as GPE)
	- Sweden (`sweden` annotated as GPE)
	- Hungary (`hungary` annotated as GPE)
	- Tunisia (`tunisia` annotated as GPE)
	- Finland (`finland` annotated as GPE)
	- Uruguay (`uruguay` annotated as GPE)
	- Austria (`austria` annotated as GPE)
	- Poland (`poland` annotated as GPE)


[('estonia', 17),
 ('norway', 16),
 ('greenlander', 11),
 ('canada', 10),
 ('czech republic', 9),
 ('france', 8),
 ('united states', 8),
 ('iceland', 8),
 ('sweden', 8),
 ('hungary', 7),
 ('tunisia', 6),
 ('finland', 6),
 ('uruguay', 6),
 ('austria', 5),
 ('poland', 5)]

#### More FN analysis

In [21]:
# Get false negatives for PHONE_NUMBER entity
fns_df = ModelError.get_fns_dataframe(results.model_errors, entity="PHONE_NUMBER")

In [22]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,60 - 56 - 85 - 91,60 56 85 91,PHONE_NUMBER,O
1,( 37 ) 788 - 063 063 966,37 788 063 063 966,PHONE_NUMBER,O
2,0490 75 40 81,0490 75 40 81,PHONE_NUMBER,O
3,0494 92 82 32,0494 92 82 32,PHONE_NUMBER,O
4,699 956 915,699 956 915,PHONE_NUMBER,O
5,( 99 ) 645 - 791,99 645 791,PHONE_NUMBER,O
6,467 3395,467 3395,PHONE_NUMBER,O
7,78 651 450,78 651 450,PHONE_NUMBER,O
8,416 60 039 +46 ( 0)8 928 571 38 +1 - 984 - 182 - 0190,416 60 039 +46 0)8 928 571 38 +1 984 182 0190,PHONE_NUMBER,O
9,0688 872 49 99,0688 872 49 99,PHONE_NUMBER,O
